In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/train-test (1).csv")
validation = pd.read_csv("../data/validation (1).csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)

train.head()

Train shape: (48000, 14)
Validation shape: (12000, 13)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [3]:
# =========================================================
# 1. Basic dataset information
# =========================================================

# Show the number of rows, columns, column types, and non-null counts
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  str    
 1   pickup        48000 non-null  str    
 2   delivery      48000 non-null  str    
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  str    
 9   weight        47700 non-null  float64
 10  date          48000 non-null  str    
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), str(5)
memory usage: 7.1 MB


In [4]:
# =========================================================
# 2. Check missing values
# =========================================================

# Count missing values in the training dataset
print("Missing values in TRAIN:")
print(train.isna().sum().sort_values(ascending=False))

print("\n" + "=" * 50 + "\n")

# Count missing values in the validation dataset
print("Missing values in VALIDATION:")
print(validation.isna().sum().sort_values(ascending=False))

Missing values in TRAIN:
market_index    374
weight          300
delivery          0
pickup_lat        0
load_id           0
pickup            0
delivery_lat      0
pickup_lon        0
distance          0
delivery_lon      0
equipment         0
date              0
quote_signal      0
posted_rate       0
dtype: int64


Missing values in VALIDATION:
market_index    249
weight          165
delivery          0
pickup            0
load_id           0
pickup_lon        0
pickup_lat        0
delivery_lat      0
delivery_lon      0
equipment         0
distance          0
date              0
quote_signal      0
dtype: int64


In [5]:
# =========================================================
# 10. Check suspicious weight values
# =========================================================

# Inspect the distribution of weight values
print(train["weight"].describe())

print("\nNegative weights in TRAIN:")
print((train["weight"] < 0).sum())

print("\nNegative weights in VALIDATION:")
print((validation["weight"] < 0).sum())

count    47700.000000
mean     31028.844004
std       9391.440620
min     -47500.000000
25%      25800.000000
50%      31436.500000
75%      37018.000000
max      47500.000000
Name: weight, dtype: float64

Negative weights in TRAIN:
292

Negative weights in VALIDATION:
145


In [6]:
# =========================================================
# 14. Investigate negative weight values
# =========================================================

# Show some rows that contain negative weights
negative_weights = train[train["weight"] < 0]

print("Sample rows with negative weights:")
display(
    negative_weights[
        ["equipment", "weight", "distance", "posted_rate"]
    ].head(10)
)

# Compare absolute negative weights with normal positive weights
print("\nAbsolute value statistics for negative weights:")
print(negative_weights["weight"].abs().describe())

print("\nPositive weight statistics:")
print(train.loc[train["weight"] > 0, "weight"].describe())

Sample rows with negative weights:


,equipment,weight,distance,posted_rate
68,Reefer,-36559.0,1334.2,2831.27
204,Dry Van,-26670.0,496.2,1046.92
206,Reefer,-20003.0,411.3,959.15
225,Reefer,-23981.0,1317.4,2836.00
376,Dry Van,-41397.0,1331.1,2522.54
446,Reefer,-37215.0,2438.3,4841.57
495,Dry Van,-31214.0,715.9,1473.78
641,Dry Van,-41023.0,582.4,1264.33
806,Dry Van,-28320.0,676.1,1307.76
1093,Dry Van,-25153.0,1058.5,1907.83



Absolute value statistics for negative weights:
count      292.000000
mean     31724.195205
std       8262.536326
min       5000.000000
25%      25928.500000
50%      31821.500000
75%      37284.250000
max      47500.000000
Name: weight, dtype: float64

Positive weight statistics:
count    47408.000000
mean     31415.358674
std       7994.902979
min       5000.000000
25%      25922.000000
50%      31493.500000
75%      37063.000000
max      47500.000000
Name: weight, dtype: float64


In [7]:
# =========================================================
# 15. Fix negative weight values
# =========================================================

# Shipment weight cannot be negative.
# The absolute values of negative weights follow nearly the
# same distribution as the valid positive weights, suggesting
# a sign-entry / encoding issue.

train["weight"] = train["weight"].abs()
validation["weight"] = validation["weight"].abs()

# Verify that no negative weights remain
print("Negative weights in TRAIN after correction:",
      (train["weight"] < 0).sum())

print("Negative weights in VALIDATION after correction:",
      (validation["weight"] < 0).sum())

Negative weights in TRAIN after correction: 0
Negative weights in VALIDATION after correction: 0


In [8]:
# =========================================================
# 16. Target variable statistics
# =========================================================

print(train["posted_rate"].describe())

count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64


In [9]:
# =========================================================
# 17. Date ranges
# =========================================================

print("Train date range:")
print(train["date"].min(), "->", train["date"].max())

print("\nValidation date range:")
print(validation["date"].min(), "->", validation["date"].max())

Train date range:
2025-01-01 -> 2025-10-31

Validation date range:
2025-11-01 -> 2025-12-31


In [11]:
# =========================================================
# 21. Time-based train/validation split
# =========================================================

# Train on earlier months
train_part = train[train["date"] < "2025-09-01"].copy()

# Validate on the most recent two months
val_part = train[train["date"] >= "2025-09-01"].copy()

print("Train period:")
print(train_part["date"].min(), "->", train_part["date"].max())

print("\nValidation period:")
print(val_part["date"].min(), "->", val_part["date"].max())

print("\nTrain rows:", len(train_part))
print("Validation rows:", len(val_part))

Train period:
2025-01-01 -> 2025-08-31

Validation period:
2025-09-01 -> 2025-10-31

Train rows: 38477
Validation rows: 9523


In [13]:
# =========================================================
# 22. Convert date columns to datetime
# =========================================================

train_part["date"] = pd.to_datetime(train_part["date"])
val_part["date"] = pd.to_datetime(val_part["date"])

print("Train date type:", train_part["date"].dtype)
print("Validation date type:", val_part["date"].dtype)

Train date type: datetime64[us]
Validation date type: datetime64[us]


In [14]:
# =========================================================
# 23. Create date-based features
# =========================================================

def add_date_features(df):
    df = df.copy()

    # Extract useful calendar features from the date column
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear

    return df


train_part = add_date_features(train_part)
val_part = add_date_features(val_part)

print("Date features added successfully.")

Date features added successfully.


In [15]:
# =========================================================
# 24. Select features and target
# =========================================================

features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear"
]

target = "posted_rate"

X_train = train_part[features]
y_train = train_part[target]

X_val = val_part[features]
y_val = val_part[target]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (38477, 13)
X_val shape: (9523, 13)
y_train shape: (38477,)
y_val shape: (9523,)


In [17]:
# =========================================================
# 25. Handle missing numeric values
# =========================================================

from sklearn.impute import SimpleImputer

# Learn median values from the training split only
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

print("Missing values handled successfully.")

Missing values handled successfully.


In [18]:
# =========================================================
# 26. Train baseline Random Forest model
# =========================================================

from sklearn.ensemble import RandomForestRegressor

baseline_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train_imputed, y_train)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [19]:
# =========================================================
# 27. Evaluate baseline model
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Generate predictions on the internal validation period
val_predictions = baseline_model.predict(X_val_imputed)

# Calculate evaluation metrics
mae = mean_absolute_error(y_val, val_predictions)
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
r2 = r2_score(y_val, val_predictions)

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

MAE:  $243.80
RMSE: $713.20
R²:   0.7816


In [20]:
# =========================================================
# 28. Prepare features for CatBoost
# =========================================================

cat_features = [
    "pickup",
    "delivery",
    "equipment"
]

model_features = [
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear"
]

X_train_cb = train_part[model_features].copy()
X_val_cb = val_part[model_features].copy()

# Fill missing numeric values using medians learned from training data
for col in ["weight", "market_index"]:
    median_value = X_train_cb[col].median()

    X_train_cb[col] = X_train_cb[col].fillna(median_value)
    X_val_cb[col] = X_val_cb[col].fillna(median_value)

print("CatBoost data prepared successfully.")
print("Train shape:", X_train_cb.shape)
print("Validation shape:", X_val_cb.shape)

CatBoost data prepared successfully.
Train shape: (38477, 16)
Validation shape: (9523, 16)


In [21]:
# =========================================================
# 29. Train CatBoost model
# =========================================================

from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=100,
    verbose=100
)

print("CatBoost model trained successfully.")

0:	learn: 1418.0512049	test: 1467.6304124	best: 1467.6304124 (0)	total: 218ms	remaining: 4m 21s
100:	learn: 573.5813781	test: 634.0202292	best: 634.0202292 (100)	total: 6.08s	remaining: 1m 6s
200:	learn: 560.7327491	test: 635.1429613	best: 633.3905349 (122)	total: 12.3s	remaining: 1m 1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 633.3905349
bestIteration = 122

Shrink model to first 123 iterations.
CatBoost model trained successfully.


In [22]:
# =========================================================
# 30. Evaluate CatBoost model
# =========================================================

catboost_predictions = catboost_model.predict(X_val_cb)

cat_mae = mean_absolute_error(y_val, catboost_predictions)
cat_rmse = np.sqrt(mean_squared_error(y_val, catboost_predictions))
cat_r2 = r2_score(y_val, catboost_predictions)

print(f"CatBoost MAE:  ${cat_mae:,.2f}")
print(f"CatBoost RMSE: ${cat_rmse:,.2f}")
print(f"CatBoost R²:   {cat_r2:.4f}")

CatBoost MAE:  $111.41
CatBoost RMSE: $633.39
CatBoost R²:   0.8277


In [23]:
# =========================================================
# 31. Prepare full training data for final model
# =========================================================

# Make sure the full datasets use datetime
train["date"] = pd.to_datetime(train["date"])
validation["date"] = pd.to_datetime(validation["date"])

# Add the same date features used during validation
train_full = add_date_features(train)
validation_full = add_date_features(validation)

# Select the same model features
X_full = train_full[model_features].copy()
y_full = train_full["posted_rate"]

X_test = validation_full[model_features].copy()

# Fill missing numeric values using medians learned from full training data
for col in ["weight", "market_index"]:
    median_value = X_full[col].median()

    X_full[col] = X_full[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

print("Full training data prepared successfully.")
print("Full train shape:", X_full.shape)
print("Final validation shape:", X_test.shape)

Full training data prepared successfully.
Full train shape: (48000, 16)
Final validation shape: (12000, 16)


In [24]:
# =========================================================
# 32. Train final CatBoost model on all development data
# =========================================================

from catboost import CatBoostRegressor

final_model = CatBoostRegressor(
    iterations=123,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=50
)

final_model.fit(
    X_full,
    y_full,
    cat_features=cat_features
)

print("Final CatBoost model trained successfully.")

0:	learn: 1428.9686044	total: 47.4ms	remaining: 5.79s
50:	learn: 605.9678268	total: 2.77s	remaining: 3.92s
100:	learn: 581.4921783	total: 5.96s	remaining: 1.3s
122:	learn: 577.3601822	total: 7.15s	remaining: 0us
Final CatBoost model trained successfully.


In [25]:
# =========================================================
# 33. Predict final validation loads
# =========================================================

final_predictions = final_model.predict(X_test)

print("Number of predictions:", len(final_predictions))
print("Minimum predicted rate:", final_predictions.min())
print("Maximum predicted rate:", final_predictions.max())

Number of predictions: 12000
Minimum predicted rate: 332.99507781898274
Maximum predicted rate: 7400.060552727492


In [26]:
# =========================================================
# 35. Show model evaluation summary
# =========================================================

print("Model Evaluation")
print("---------------------------")
print(f"MAE  : ${cat_mae:,.2f}")
print(f"RMSE : ${cat_rmse:,.2f}")
print(f"R²   : {cat_r2:.4f}")
print(f"R² % : {cat_r2 * 100:.2f}%")

Model Evaluation
---------------------------
MAE  : $111.41
RMSE : $633.39
R²   : 0.8277
R² % : 82.77%


In [27]:
# =========================================================
# 36. Add route feature
# =========================================================

# Create a single categorical feature representing the lane
train_part["route"] = (
    train_part["pickup"].astype(str)
    + "_TO_"
    + train_part["delivery"].astype(str)
)

val_part["route"] = (
    val_part["pickup"].astype(str)
    + "_TO_"
    + val_part["delivery"].astype(str)
)

print("Route feature added successfully.")
print("Unique train routes:", train_part["route"].nunique())
print("Unique validation routes:", val_part["route"].nunique())

Route feature added successfully.
Unique train routes: 3999
Unique validation routes: 3337


In [28]:
# =========================================================
# 37. Prepare CatBoost data with route feature
# =========================================================

cat_features_v2 = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

model_features_v2 = [
    "pickup",
    "delivery",
    "route",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear"
]

X_train_cb_v2 = train_part[model_features_v2].copy()
X_val_cb_v2 = val_part[model_features_v2].copy()

# Fill missing numeric values using medians from the training split
for col in ["weight", "market_index"]:
    median_value = X_train_cb_v2[col].median()

    X_train_cb_v2[col] = X_train_cb_v2[col].fillna(median_value)
    X_val_cb_v2[col] = X_val_cb_v2[col].fillna(median_value)

print("Updated CatBoost data prepared successfully.")
print("Train shape:", X_train_cb_v2.shape)
print("Validation shape:", X_val_cb_v2.shape)

Updated CatBoost data prepared successfully.
Train shape: (38477, 17)
Validation shape: (9523, 17)


In [29]:
# =========================================================
# 38. Train CatBoost model with route feature
# =========================================================

catboost_model_v2 = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model_v2.fit(
    X_train_cb_v2,
    y_train,
    cat_features=cat_features_v2,
    eval_set=(X_val_cb_v2, y_val),
    early_stopping_rounds=120,
    verbose=100
)

print("CatBoost v2 trained successfully.")

0:	learn: 1429.7071276	test: 1479.3254541	best: 1479.3254541 (0)	total: 57.3ms	remaining: 1m 25s
100:	learn: 579.1571865	test: 635.8659410	best: 635.8659410 (100)	total: 5.79s	remaining: 1m 20s
200:	learn: 566.0438085	test: 633.5548255	best: 633.0567769 (172)	total: 11.3s	remaining: 1m 13s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 633.0567769
bestIteration = 172

Shrink model to first 173 iterations.
CatBoost v2 trained successfully.


In [30]:
# =========================================================
# 39. Evaluate CatBoost v2
# =========================================================

catboost_predictions_v2 = catboost_model_v2.predict(X_val_cb_v2)

cat_mae_v2 = mean_absolute_error(y_val, catboost_predictions_v2)
cat_rmse_v2 = np.sqrt(mean_squared_error(y_val, catboost_predictions_v2))
cat_r2_v2 = r2_score(y_val, catboost_predictions_v2)

print(f"CatBoost v2 MAE:  ${cat_mae_v2:,.2f}")
print(f"CatBoost v2 RMSE: ${cat_rmse_v2:,.2f}")
print(f"CatBoost v2 R²:   {cat_r2_v2:.4f}")

CatBoost v2 MAE:  $114.58
CatBoost v2 RMSE: $633.06
CatBoost v2 R²:   0.8279


In [31]:
# =========================================================
# 40. Add interaction features
# =========================================================

train_part["distance_market"] = (
    train_part["distance"] * train_part["market_index"]
)

val_part["distance_market"] = (
    val_part["distance"] * val_part["market_index"]
)

train_part["distance_quote"] = (
    train_part["distance"] * train_part["quote_signal"]
)

val_part["distance_quote"] = (
    val_part["distance"] * val_part["quote_signal"]
)

train_part["weight_per_mile"] = (
    train_part["weight"] / train_part["distance"]
)

val_part["weight_per_mile"] = (
    val_part["weight"] / val_part["distance"]
)

print("Interaction features added successfully.")

Interaction features added successfully.


In [32]:
# =========================================================
# 41. Prepare CatBoost data with interaction features
# =========================================================

cat_features_v3 = [
    "pickup",
    "delivery",
    "equipment"
]

model_features_v3 = [
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "distance_market",
    "distance_quote",
    "weight_per_mile"
]

X_train_cb_v3 = train_part[model_features_v3].copy()
X_val_cb_v3 = val_part[model_features_v3].copy()

# Fill missing numeric values using medians from the training split
for col in ["weight", "market_index", "distance_market", "weight_per_mile"]:
    median_value = X_train_cb_v3[col].median()

    X_train_cb_v3[col] = X_train_cb_v3[col].fillna(median_value)
    X_val_cb_v3[col] = X_val_cb_v3[col].fillna(median_value)

print("CatBoost v3 data prepared successfully.")
print("Train shape:", X_train_cb_v3.shape)
print("Validation shape:", X_val_cb_v3.shape)

CatBoost v3 data prepared successfully.
Train shape: (38477, 19)
Validation shape: (9523, 19)


In [33]:
# =========================================================
# 42. Train CatBoost v3 with interaction features
# =========================================================

catboost_model_v3 = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model_v3.fit(
    X_train_cb_v3,
    y_train,
    cat_features=cat_features_v3,
    eval_set=(X_val_cb_v3, y_val),
    early_stopping_rounds=120,
    verbose=100
)

print("CatBoost v3 trained successfully.")

0:	learn: 1429.7931510	test: 1478.9761187	best: 1478.9761187 (0)	total: 51ms	remaining: 1m 16s
100:	learn: 576.2168044	test: 641.5519670	best: 641.5519670 (100)	total: 5.32s	remaining: 1m 13s
200:	learn: 560.2770429	test: 636.2200383	best: 636.1576452 (190)	total: 10.3s	remaining: 1m 6s
300:	learn: 550.2861779	test: 636.1175271	best: 635.7690432 (276)	total: 15.2s	remaining: 1m
Stopped by overfitting detector  (120 iterations wait)

bestTest = 635.7690432
bestIteration = 276

Shrink model to first 277 iterations.
CatBoost v3 trained successfully.


In [34]:
# =========================================================
# 43. Evaluate CatBoost v3
# =========================================================

catboost_predictions_v3 = catboost_model_v3.predict(X_val_cb_v3)

cat_mae_v3 = mean_absolute_error(y_val, catboost_predictions_v3)
cat_rmse_v3 = np.sqrt(mean_squared_error(y_val, catboost_predictions_v3))
cat_r2_v3 = r2_score(y_val, catboost_predictions_v3)

print(f"CatBoost v3 MAE:  ${cat_mae_v3:,.2f}")
print(f"CatBoost v3 RMSE: ${cat_rmse_v3:,.2f}")
print(f"CatBoost v3 R²:   {cat_r2_v3:.4f}")

CatBoost v3 MAE:  $120.23
CatBoost v3 RMSE: $635.77
CatBoost v3 R²:   0.8264


In [35]:
# =========================================================
# 44. Train CatBoost v4 with deeper trees
# =========================================================

catboost_model_v4 = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=10,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model_v4.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=150,
    verbose=100
)

print("CatBoost v4 trained successfully.")

0:	learn: 1441.0485710	test: 1489.8592163	best: 1489.8592163 (0)	total: 76.4ms	remaining: 2m 32s
100:	learn: 579.8919268	test: 642.2647453	best: 642.2647453 (100)	total: 7.86s	remaining: 2m 27s
200:	learn: 557.5126692	test: 633.9374824	best: 633.9374824 (200)	total: 16.6s	remaining: 2m 28s
300:	learn: 547.1694163	test: 634.0300308	best: 633.6103470 (257)	total: 25.1s	remaining: 2m 21s
400:	learn: 540.7388098	test: 634.5724611	best: 633.6103470 (257)	total: 33s	remaining: 2m 11s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 633.610347
bestIteration = 257

Shrink model to first 258 iterations.
CatBoost v4 trained successfully.


In [36]:
# =========================================================
# 45. Prepare XGBoost preprocessing pipeline
# =========================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_features_xgb = [
    "pickup",
    "delivery",
    "equipment"
]

numeric_features_xgb = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_xgb),
        ("cat", categorical_transformer, categorical_features_xgb)
    ]
)

print("XGBoost preprocessing pipeline created successfully.")

XGBoost preprocessing pipeline created successfully.


In [37]:
# =========================================================
# 46. Transform data for XGBoost
# =========================================================

X_train_xgb = preprocessor_xgb.fit_transform(train_part)
X_val_xgb = preprocessor_xgb.transform(val_part)

print("XGBoost data transformed successfully.")
print("Train shape:", X_train_xgb.shape)
print("Validation shape:", X_val_xgb.shape)

XGBoost data transformed successfully.
Train shape: (38477, 144)
Validation shape: (9523, 144)


In [38]:
# =========================================================
# 47. Train XGBoost model
# =========================================================

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=100
)

print("XGBoost model trained successfully.")

[0]	validation_0-rmse:1489.28959
[100]	validation_0-rmse:653.65873
[200]	validation_0-rmse:658.41819
[300]	validation_0-rmse:667.83246
[400]	validation_0-rmse:672.02083
[500]	validation_0-rmse:675.75043
[600]	validation_0-rmse:678.29471
[700]	validation_0-rmse:679.99931
[800]	validation_0-rmse:682.03624
[900]	validation_0-rmse:683.45735
[1000]	validation_0-rmse:684.42391
[1100]	validation_0-rmse:685.38056
[1200]	validation_0-rmse:686.44490
[1300]	validation_0-rmse:687.16520
[1400]	validation_0-rmse:687.97729
[1499]	validation_0-rmse:688.59297
XGBoost model trained successfully.


In [39]:
# =========================================================
# 48. Find the best XGBoost iteration
# =========================================================

results = xgb_model.evals_result()

rmse_values = results["validation_0"]["rmse"]

best_iteration = int(np.argmin(rmse_values))
best_rmse = rmse_values[best_iteration]

print("Best XGBoost iteration:", best_iteration)
print(f"Best validation RMSE: ${best_rmse:,.2f}")

Best XGBoost iteration: 124
Best validation RMSE: $652.87


In [40]:
# =========================================================
# 49. Retrain XGBoost using the best number of trees
# =========================================================

xgb_best = XGBRegressor(
    n_estimators=125,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_best.fit(
    X_train_xgb,
    y_train
)

xgb_predictions = xgb_best.predict(X_val_xgb)

xgb_mae = mean_absolute_error(y_val, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_predictions))
xgb_r2 = r2_score(y_val, xgb_predictions)

print(f"XGBoost MAE:  ${xgb_mae:,.2f}")
print(f"XGBoost RMSE: ${xgb_rmse:,.2f}")
print(f"XGBoost R²:   {xgb_r2:.4f}")

XGBoost MAE:  $157.51
XGBoost RMSE: $652.87
XGBoost R²:   0.8170


In [41]:
# =========================================================
# 50. Compare model performance
# =========================================================

results_df = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "CatBoost"
    ],
    "MAE": [
        mae,
        xgb_mae,
        cat_mae
    ],
    "RMSE": [
        rmse,
        xgb_rmse,
        cat_rmse
    ],
    "R2": [
        r2,
        xgb_r2,
        cat_r2
    ]
})

results_df = results_df.sort_values("MAE").reset_index(drop=True)

display(results_df)

,Model,MAE,RMSE,R2
0,CatBoost,111.411603,633.390540,0.827732
1,XGBoost,157.507983,652.868621,0.816974
2,Random Forest,243.799929,713.204333,0.781582


In [42]:
# =========================================================
# 51. Try a weighted ensemble
# =========================================================

ensemble_predictions = (
    0.85 * catboost_predictions +
    0.15 * xgb_predictions
)

ensemble_mae = mean_absolute_error(y_val, ensemble_predictions)
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_predictions))
ensemble_r2 = r2_score(y_val, ensemble_predictions)

print(f"Ensemble MAE:  ${ensemble_mae:,.2f}")
print(f"Ensemble RMSE: ${ensemble_rmse:,.2f}")
print(f"Ensemble R²:   {ensemble_r2:.4f}")

Ensemble MAE:  $112.06
Ensemble RMSE: $633.44
Ensemble R²:   0.8277


In [45]:
# =========================================================
# 52. Prepare December data for prediction
# =========================================================

december = pd.read_csv("../data/december-chart-inputs (2).csv")

# Shipment weight should always be positive
december["weight"] = december["weight"].abs()

# Convert date to datetime
december["date"] = pd.to_datetime(december["date"])

# Add the same date features used during training
december_full = add_date_features(december)

# Select the same features used by the final CatBoost model
X_december = december_full[model_features].copy()

# Fill missing values using medians learned from the full training data
for col in ["weight", "market_index"]:
    median_value = X_full[col].median()
    X_december[col] = X_december[col].fillna(median_value)

print("December data prepared successfully.")
print("Shape:", X_december.shape)

KeyError: "['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'market_index', 'quote_signal'] not in index"

In [46]:
# =========================================================
# 53. Recover missing December features from training data
# =========================================================

# Get historical rows for the same fixed lane
lane_history = train[
    (train["pickup"] == "Lexington") &
    (train["delivery"] == "Fort Wayne")
].copy()

print("Historical rows for Lexington -> Fort Wayne:", len(lane_history))

if len(lane_history) > 0:
    print("\nReference values:")
    print("pickup_lat:", lane_history["pickup_lat"].median())
    print("pickup_lon:", lane_history["pickup_lon"].median())
    print("delivery_lat:", lane_history["delivery_lat"].median())
    print("delivery_lon:", lane_history["delivery_lon"].median())
    print("market_index median:", lane_history["market_index"].median())
    print("quote_signal median:", lane_history["quote_signal"].median())

Historical rows for Lexington -> Fort Wayne: 32

Reference values:
pickup_lat: 36.99152
pickup_lon: -84.99876
delivery_lat: 41.31561
delivery_lon: -85.36206
market_index median: 1.0440200000000002
quote_signal median: 1.97252


In [47]:
# =========================================================
# 54. Add missing model features to December data
# =========================================================

# Use historical information from the same lane
december_full["pickup_lat"] = lane_history["pickup_lat"].median()
december_full["pickup_lon"] = lane_history["pickup_lon"].median()

december_full["delivery_lat"] = lane_history["delivery_lat"].median()
december_full["delivery_lon"] = lane_history["delivery_lon"].median()

december_full["market_index"] = lane_history["market_index"].median()
december_full["quote_signal"] = lane_history["quote_signal"].median()

# Now select the exact same features used by the final model
X_december = december_full[model_features].copy()

print("December model features prepared successfully.")
print("Shape:", X_december.shape)

display(X_december.head())

December model features prepared successfully.
Shape: (31, 16)


,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,market_index,quote_signal,year,month,day,dayofweek,dayofyear
0,Lexington,Fort Wayne,36.99152,-84.99876,41.31561,-85.36206,360,Dry Van,32000,1.04402,1.97252,2025,12,1,0,335
1,Lexington,Fort Wayne,36.99152,-84.99876,41.31561,-85.36206,360,Dry Van,32000,1.04402,1.97252,2025,12,2,1,336
2,Lexington,Fort Wayne,36.99152,-84.99876,41.31561,-85.36206,360,Dry Van,32000,1.04402,1.97252,2025,12,3,2,337
3,Lexington,Fort Wayne,36.99152,-84.99876,41.31561,-85.36206,360,Dry Van,32000,1.04402,1.97252,2025,12,4,3,338
4,Lexington,Fort Wayne,36.99152,-84.99876,41.31561,-85.36206,360,Dry Van,32000,1.04402,1.97252,2025,12,5,4,339


In [48]:
# =========================================================
# 55. Predict and save December predictions
# =========================================================

# Predict rates for all 31 December days
december_predictions = final_model.predict(X_december)

# Fill the original December file
december["predicted_rate"] = december_predictions

# Keep the exact column order required by score.py
december_output = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
].copy()

# Save the completed December file
december_output.to_csv(
    "../data/december_chart_inputs.csv",
    index=False
)

print("December predictions saved successfully.")
print("Shape:", december_output.shape)

display(december_output.head())

December predictions saved successfully.
Shape: (31, 7)


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,849.263704
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,852.616206
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,853.167217
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,853.737364
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,853.196036


In [1]:
# =========================================================
# Save validation_predictions.csv in project root
# =========================================================

submission = pd.DataFrame({
    "load_id": validation["load_id"],
    "predicted_rate": final_predictions
})

submission.to_csv(
    "../validation_predictions.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", submission.shape)

NameError: name 'pd' is not defined

In [2]:
# =========================================================
# Restore required variables
# =========================================================

import pandas as pd
import numpy as np

# Reload validation data
validation = pd.read_csv("../data/validation (1).csv")

print("Validation shape:", validation.shape)

# Check whether final_predictions still exists
print("final_predictions exists:", "final_predictions" in globals())

Validation shape: (12000, 13)
final_predictions exists: False


In [3]:
# =========================================================
# Rebuild final CatBoost model and predictions
# =========================================================

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv("../data/train-test (1).csv")
validation = pd.read_csv("../data/validation (1).csv")

# Clean negative weights
train["weight"] = train["weight"].abs()
validation["weight"] = validation["weight"].abs()

# Convert dates
train["date"] = pd.to_datetime(train["date"])
validation["date"] = pd.to_datetime(validation["date"])

# Date feature function
def add_date_features(df):
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    return df

train_full = add_date_features(train)
validation_full = add_date_features(validation)

# Model features
model_features = [
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear"
]

cat_features = ["pickup", "delivery", "equipment"]

X_full = train_full[model_features].copy()
y_full = train_full["posted_rate"]

X_test = validation_full[model_features].copy()

# Fill missing values using training medians
for col in ["weight", "market_index"]:
    median_value = X_full[col].median()
    X_full[col] = X_full[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

# Train final model
final_model = CatBoostRegressor(
    iterations=123,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=50
)

final_model.fit(
    X_full,
    y_full,
    cat_features=cat_features
)

# Predict
final_predictions = final_model.predict(X_test)

print("Predictions created:", len(final_predictions))

0:	learn: 1428.9686044	total: 195ms	remaining: 23.8s
50:	learn: 605.9678268	total: 2.93s	remaining: 4.14s
100:	learn: 581.4921783	total: 6.48s	remaining: 1.41s
122:	learn: 577.3601822	total: 8.03s	remaining: 0us
Predictions created: 12000


In [4]:
# =========================================================
# Save final validation predictions
# =========================================================

submission = pd.DataFrame({
    "load_id": validation["load_id"],
    "predicted_rate": final_predictions
})

submission.to_csv(
    "../validation_predictions.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", submission.shape)

display(submission.head())

Saved successfully.
Shape: (12000, 2)


,load_id,predicted_rate
0,TE-000001,871.757127
1,TE-000002,4968.559160
2,TE-000003,5229.743285
3,TE-000004,3960.341576
4,TE-000005,1897.805221
